In [ ]:
import ctypes
import math

class DynamicArray:
    def __init__(self):
        self.__size = 0
        self.__capacity = 1
        self.__array = self.__allocate(self.__capacity)

    def __allocate(self, capacity):
        return (ctypes.py_object * capacity)()

    def len(self):
        return self.__size

    def iter_seq(self):
        for i in range(self.__size):
            yield self.__array[i]

    def build(self, X):
        self.__size = len(X)

        if self.__size == 0:
            self.__capacity = 1
        else:
            self.__capacity = 2**math.ceil(math.log2(self.__size))
        
        self.__array = self.__allocate(self.__capacity)
        
        for i in range(self.__size):
            self.__array[i] = X[i]

    def get_at(self, i):
        if i < 0 or i >= self.__size:
            raise IndexError

        return self.__array[i]

    def set_at(self, i, x):
        if i < 0 or i >= self.__size:
            raise IndexError

        self.__array[i] = x

    def __resize(self, new_capacity, copy_count):
        temp_array = self.__array
        self.__array = self.__allocate(new_capacity)
        for i in range(copy_count):
            self.__array[i] = temp_array[i]
        self.__capacity = new_capacity

    def __grow_if_needed(self):
        if self.__size == self.__capacity:
            self.__resize(2**math.ceil(math.log2(self.__size + 1)), self.__size)

    def __shrink_if_needed(self):
        if self.__capacity > 1 and (self.__size - 1) * 4 <= self.__capacity:
            self.__resize(max(1, self.__capacity // 2), self.__size - 1)

    def insert_last(self, x):
        self.__grow_if_needed()
        self.__array[self.__size] = x
        self.__size += 1

    def delete_last(self):
        if self.__size == 0:
            raise IndexError
        
        value = self.__array[self.__size - 1]
        self.__array[self.__size - 1] = None
        self.__shrink_if_needed()
        self.__size -= 1
        return value

    def insert_first(self, x):
        self.__grow_if_needed()
        for i in range(self.__size - 1, -1, -1):
            self.__array[i+1] = self.__array[i]
        self.__array[0] = x
        self.__size += 1
        
    def delete_first(self):
        if self.__size == 0:
            raise IndexError

        value = self.__array[0]

        for i in range(1, self.__size, 1):
            self.__array[i - 1] = self.__array[i]
        self.__array[self.__size - 1] = None

        self.__shrink_if_needed()

        self.__size -= 1
        return value

    def insert_at(self, i, x):
        if i < 0 or i > self.__size:
            raise IndexError
        elif i == 0:
            self.insert_first(x)
            return
        elif i == self.__size:
            self.insert_last(x)
            return
        
        self.__grow_if_needed()
        for j in range(self.__size, i, -1):
            self.__array[j] = self.__array[j-1]
        self.__array[i] = x
        self.__size += 1

    def delete_at(self, i):
        if i < 0 or i >= self.__size:
            raise IndexError
        elif i == 0:
            return self.delete_first()
        elif i == self.__size - 1:
            return self.delete_last()
            
        value = self.__array[i]
        for j in range(i, self.__size -1):
            self.__array[j] = self.__array[j+1]
        self.__array[self.__size - 1] = None
        self.__shrink_if_needed()
        self.__size -= 1

        return value

